[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_he_en_colab_validation.ipynb)

Open this exact notebook in Colab, connect to a runtime, and run cells from the top using ▶. GitHub Preview displays the notebook without executing it.


# LaBSE — Hebrew–English employer-name equivalence probe

**Purpose:** Test whether the *pretrained* LaBSE model assigns higher cross-lingual similarity to corresponding organization names than to unrelated organizations. No retraining, paid API, token, GPU, or manually maintained dictionary is required. Run every cell in **Google Colab (free CPU)**.

**Limitations:** LaBSE is trained for cross-language sentence representations, **not** verified employer entity resolution. This notebook reports observed cosine similarities and nearest neighbors. A high score is not proof that names refer to the same legal entity. Real data integration requires validation. The first execution downloads publicly available model weights (~1.9 GB) and requires Internet access. No model inference result is claimed until this notebook executes.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2"])

import torch
from sentence_transformers import SentenceTransformer
import numpy as np

print("PyTorch:", torch.__version__)
print("Device: CPU (no GPU required)")

In [ ]:
from IPython.display import HTML, display
from html import escape
import numpy as np

# Data examples are the original test inputs. Extra Hebrew forms below are
# diagnostic candidates, NOT additional observed values from the survey.
original_labels = [
    "Intel", "אינטל", "NVIDIA", "נבידיה",
    "Microsoft", "מיקרוסופט", "bird", "birds", "ציפור",
    "קרית גת", "קריית גת", "בית ספר יסודי", "primary school"
]
alternate_nvidia_spellings = ["אנבידיה", "אנווידיה"]
labels = original_labels + alternate_nvidia_spellings

torch.set_num_threads(2)
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
vectors = model.encode(
    labels, batch_size=4, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False
)
assert vectors.shape == (len(labels), 768), vectors.shape
assert np.isfinite(vectors).all()
positions = {label: i for i, label in enumerate(labels)}

def cosine(a, b):
    return float(vectors[positions[a]] @ vectors[positions[b]])

def show_table(heading, rows, caption=None):
    html_rows = []
    for left, right, description in rows:
        score = cosine(left, right)
        html_rows.append(
            "<tr>"
            f"<td>{escape(left)}</td><td>{escape(right)}</td>"
            f"<td style='text-align:right;font-variant-numeric:tabular-nums'>{score:.4f}</td>"
            f"<td>{escape(description)}</td>"
            "</tr>"
        )
    table = (
        f"<h3>{escape(heading)}</h3>"
        "<table style='border-collapse:collapse;width:100%;font-size:14px'>"
        "<thead><tr>"
        "<th style='text-align:left'>Name A</th><th style='text-align:left'>Name B</th>"
        "<th style='text-align:right'>Cosine similarity</th>"
        "<th style='text-align:left'>Known relationship</th>"
        "</tr></thead><tbody>"
        + "".join(html_rows) +
        "</tbody></table>"
    )
    if caption:
        table += f"<p><small>{escape(caption)}</small></p>"
    display(HTML(table))

display(HTML(
    "<h2>LaBSE: cross-language name matching</h2>"
    f"<p>Successfully encoded <b>{len(labels)} names</b> into "
    f"<b>{vectors.shape[1]}-dimensional vectors</b> using pretrained PyTorch LaBSE."
    " Similarity is a cosine score, <b>not a probability</b> of identity.</p>"
))

show_table("1. Known equivalent names", [
    ("Intel", "אינטל", "Same company"),
    ("Microsoft", "מיקרוסופט", "Same company"),
    ("NVIDIA", "נבידיה", "Same company; spelling present in survey")
], "These are positive examples. The model's scores do not define a universal match threshold.")

show_table("2. Different names and linguistic controls", [
    ("NVIDIA", "Intel", "Different companies, similar industry"),
    ("Intel", "בית ספר יסודי", "Unrelated entities"),
    ("bird", "birds", "Same concept, singular/plural"),
    ("bird", "ציפור", "Translation"),
    ("קרית גת", "קריית גת", "Same city, spelling variants"),
    ("בית ספר יסודי", "primary school", "Translation")
])

show_table("3. Why is NVIDIA lower? Test alternative Hebrew spellings", [
    ("NVIDIA", "נבידיה", "Survey spelling"),
    ("NVIDIA", "אנבידיה", "Alternative Hebrew spelling; not in survey test data"),
    ("NVIDIA", "אנווידיה", "Alternative Hebrew spelling; not in survey test data")
], "Compare these measured scores to test a spelling-sensitivity hypothesis. A difference alone does not establish the model's internal cause.")

nvidia_alias = cosine("NVIDIA", "נבידיה")
hard_negative = cosine("NVIDIA", "Intel")
candidates = [(spelling, cosine("NVIDIA", spelling)) for spelling in ["נבידיה", "אנבידיה", "אנווידיה"]]
top_spelling, top_score = max(candidates, key=lambda x: x[1])

display(HTML(
    "<h3>4. What the test actually establishes</h3>"
    f"<p>The provided Hebrew alias for NVIDIA scored <b>{nvidia_alias:.4f}</b>, "
    f"compared with <b>{hard_negative:.4f}</b> against Intel, a different company. "
    f"The observed margin is <b>{nvidia_alias-hard_negative:+.4f}</b>.</p>"
    f"<p>The closest of the three tested Hebrew spellings is "
    f"<b>{escape(top_spelling)}</b> (score <b>{top_score:.4f}</b>). "
    "A higher similarity for another spelling would support spelling sensitivity; "
    "it would <b>not</b> prove a particular cause or guarantee correct entity resolution.</p>"
    "<p><b>Decision:</b> This is a diagnostic experiment, not a verified production "
    "matching algorithm. Do not automatically merge employer names from these scores alone.</p>"
))
print("MODEL_EXECUTION_SUCCESS: All similarities above were measured during this run.")


**Interpretation rule:** A successful run proves that the pipeline produces real embeddings and similarity scores. It does **not** establish the precision of multilingual company-name matching. Decide whether to use this representation only after evaluating positive and negative pairs from the actual `ShemAvoda` distribution, including ambiguous names and generic employer descriptions. The notebook intentionally performs no automatic name replacement.